In [1]:
import polars as pl

# Load lazily to reduce RAM usage
df = pl.read_parquet("../data/malaysia_transactions.parquet")

In [2]:
columns_to_keep = [
    "date_time",
    "ofi_entity_id",
    "rfi_entity_id",
    "trxn_amount",
    "trxn_type",
    "trxn_channel"
]

df_clean = df.select(columns_to_keep)

df_clean.null_count()

date_time,ofi_entity_id,rfi_entity_id,trxn_amount,trxn_type,trxn_channel
u32,u32,u32,u32,u32,u32
0,438281,464000,0,665434,0


In [3]:
# Drop any rows with missing sender/receiver IDs
df_filtered = df_clean.filter(
    pl.col("ofi_entity_id").is_not_null() & 
    pl.col("rfi_entity_id").is_not_null()
)

# Fill missing `trxn_type` with fallback label
df_filtered = df_filtered.with_columns(
    pl.col("trxn_type").fill_null("Unknown")
)

In [4]:
df_filtered.null_count()

date_time,ofi_entity_id,rfi_entity_id,trxn_amount,trxn_type,trxn_channel
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


In [5]:
df_filtered.shape

(11396551, 6)

In [6]:
# Sort the dataframe by time
df_sorted = df_filtered.sort("date_time")

# check earliest and latest timestamp
print("Start:", df_sorted["date_time"][0])
print("End:", df_sorted["date_time"][-1])

Start: 2025-06-01 00:00:00
End: 2025-06-30 23:59:59


In [7]:
from collections import defaultdict

graph = defaultdict(list)
incoming_graph = defaultdict(list)
entity_risk = defaultdict(float)
last_incoming_time = {}

In [8]:
from datetime import datetime, timedelta

# Rule threshold
MAX_LAYERING_DEPTH = 4
MAX_TIME_GAP = timedelta(minutes=10)  # fast flow between nodes

# Utility: parse timestamp if it's not already datetime
def ensure_datetime(ts):
    return ts if isinstance(ts, datetime) else datetime.strptime(str(ts), "%Y-%m-%d %H:%M:%S")

# Recursive DFS to measure path depth
def layering_depth(entity, visited, current_time, depth=0):
    if depth >= MAX_LAYERING_DEPTH:
        return depth

    max_depth = depth
    for (target, t, amt, ttype) in graph[entity]:
        t = ensure_datetime(t)
        if target not in visited and (current_time - t) <= MAX_TIME_GAP:
            visited.add(target)
            new_depth = layering_depth(target, visited, t, depth + 1)
            max_depth = max(max_depth, new_depth)
            visited.remove(target)

    return max_depth

def process_transaction(sender, receiver, timestamp, amount, ttype):
    timestamp = ensure_datetime(timestamp)

    # Store in graphs
    graph[sender].append((receiver, timestamp, amount, ttype))
    incoming_graph[receiver].append((sender, timestamp, amount, ttype))

    # Compute layering depth
    visited = set([sender])
    depth = layering_depth_backtrace(sender, visited, timestamp)

    # if depth >= 1:
    # Time delta
    prev_time = last_incoming_time.get(receiver)
    last_incoming_time[receiver] = timestamp  # always update first

    # now compute time gap
    time_gap = (timestamp - prev_time).total_seconds() / 60 if prev_time else 8

    print(f"timestamp {sender} - prev time {prev_time}")
    print(f"Layering depth: {depth}, Time gap: {time_gap:.2f} minutes")

    # Normalized risk boost (max 0.8, fades with time)
    thres = 10.0
    base_risk_boost = max(0.0, (thres - time_gap) / thres)
    new_risk = 0.2 * entity_risk[receiver] + 0.8 * base_risk_boost
    print(f"Risk[{new_risk}]: Processing transaction from {sender} to {receiver} at {timestamp} with amount {amount}")

    # Update risk
    entity_risk[receiver] = new_risk

    if new_risk >= 0.9:
        print(f"⚠️ Transaction blocked: Risk too high for {receiver} ({new_risk:.2f})")
        return new_risk# stop processing

    # Track last incoming time
    last_incoming_time[receiver] = timestamp
    return new_risk


def layering_depth_backtrace(entity, visited, current_time, depth=0):
    if depth >= MAX_LAYERING_DEPTH:
        return depth

    max_depth = depth
    for (source, t, amt, ttype) in incoming_graph[entity]:
        t = ensure_datetime(t)
        if source not in visited and (current_time - t) <= MAX_TIME_GAP:
            visited.add(source)
            new_depth = layering_depth_backtrace(source, visited, t, depth + 1)
            max_depth = max(max_depth, new_depth)
            visited.remove(source)

    return max_depth

In [9]:
from datetime import datetime, timedelta

base_time = datetime(2025, 6, 1, 12, 0, 0)
synthetic_fan_in = [
    ("E001", "E100", base_time, 1000.0, "Online Transfer"),
    ("E002", "E100", base_time + timedelta(seconds=30), 1000.0, "Online Transfer"),
    ("E003", "E100", base_time + timedelta(seconds=30), 1000.0, "Online Transfer"),
]

for s, r, t, a, tt in synthetic_fan_in:
    return_risk = process_transaction(s, r, t, a, tt) 
    if return_risk < 0.9:
        print(f"return risk:{return_risk}\n")
    else:
        break

timestamp E001 - prev time None
Layering depth: 0, Time gap: 8.00 minutes
Risk[0.16000000000000003]: Processing transaction from E001 to E100 at 2025-06-01 12:00:00 with amount 1000.0
return risk:0.16000000000000003

timestamp E002 - prev time 2025-06-01 12:00:00
Layering depth: 0, Time gap: 0.50 minutes
Risk[0.792]: Processing transaction from E002 to E100 at 2025-06-01 12:00:30 with amount 1000.0
return risk:0.792

timestamp E003 - prev time 2025-06-01 12:00:30
Layering depth: 0, Time gap: 0.00 minutes
Risk[0.9584]: Processing transaction from E003 to E100 at 2025-06-01 12:00:30 with amount 1000.0
⚠️ Transaction blocked: Risk too high for E100 (0.96)


In [ ]:
# from datetime import datetime, timedelta

# base_time = datetime(2025, 6, 1, 12, 0, 0)

# for i in range(10):
#     s = f"E{i:03}"
#     r = f"E{i+1:03}"
#     t = base_time + timedelta(minutes=i)  # try shorter gap for first few
#     a = 1000.0 - i * 10
#     process_transaction(s, r, t, a, "Online Transfer")

In [ ]:
# print("Risk on E002:", entity_risk["E002"])
# print("Risk on E003:", entity_risk["E003"])
# print("Risk on E004:", entity_risk["E004"])
# print("Risk on E005:", entity_risk["E005"])

In [ ]:
# from datetime import datetime, timedelta

# base_time = datetime(2025, 6, 1, 12, 0, 0)

# synthetic_chain = [
#     ("E001", "E002", base_time, 1000.0, "Online Transfer"),
#     ("E002", "E003", base_time + timedelta(minutes=1), 980.0, "Online Transfer"),
#     ("E003", "E004", base_time + timedelta(minutes=2), 970.0, "Online Transfer"),
#     ("E004", "E005", base_time + timedelta(minutes=3), 950.0, "Online Transfer"),
# ]

# for s, r, t, a, tt in synthetic_chain:
#     process_transaction(s, r, t, a, tt)

# print("Risk on E002:", entity_risk["E002"])
# print("Risk on E003:", entity_risk["E003"])
# print("Risk on E004:", entity_risk["E004"])
# print("Risk on E005:", entity_risk["E005"])

In [9]:
# row = df_sorted.row(0)
# process_transaction(
#     sender=row[1],          # ofi_entity_id
#     receiver=row[2],        # rfi_entity_id
#     timestamp=row[0],       # date_time
#     amount=row[3],          # trxn_amount
#     ttype=row[4]            # trxn_type
# )

In [ ]:
# N = 100000  # start small, increase later if fast

# for i in range(N):
#     row = df_sorted.row(i)
#     process_transaction(
#         sender=row[1],
#         receiver=row[2],
#         timestamp=row[0],
#         amount=row[3],
#         ttype=row[4]
#     )

# # Inspect: how many entities have non-zero risk
# non_zero_risk = {k: v for k, v in entity_risk.items() if v > 0}
# print("Entities with non-zero risk:", len(non_zero_risk))

# # Show top 10 riskiest entities
# top_risky = sorted(non_zero_risk.items(), key=lambda x: x[1], reverse=True)[:10]
# print("Top risky entities:", top_risky)